# 05 -- SSH et automation

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- comprendre les bases de SSH (cles, auth) ;
- automatiser des commandes distantes avec `paramiko` ;
- utiliser `fabric` pour des workflows declaratifs ;
- transferer des fichiers avec SFTP ;
- gerer les contextes (sudo, cd, env) ;
- executer en parallele sur plusieurs hotes ;
- securiser la gestion des cles.

## Prerequis -- ce que vous devez maitriser

- contextes managers et decorateurs ;
- bases systeme Linux ;
- clefs SSH (generation avec `ssh-keygen`).

Ce que ce notebook **ne couvrira pas** :

- Ansible (orchestration declarative plus riche) ;
- SaltStack, Puppet, Chef ;
- cluster management Kubernetes.

## Plan

1. SSH rappels
2. Installation des bibliotheques
3. Paramiko : premiere connexion
4. Executer une commande
5. SFTP : transfert de fichiers
6. Gestion de cles
7. Fabric : connexions et groupes
8. Sudo et context managers
9. Parallelisme
10. Gestion d'erreurs
11. Logging et secrets
12. Synthese
13. Exercices

---

## 1. SSH rappels

- **SSH** (Secure Shell) : protocole chiffre pour shell/fichiers distants
- **Cles** : paire privee/publique generee par `ssh-keygen`
- Auth par **mot de passe**, **cle publique**, ou **agent**
- Config locale : `~/.ssh/config`
- Agent : `ssh-add ~/.ssh/id_ed25519`

---

## 2. Installation des bibliotheques

In [ ]:
# pip install paramiko fabric
import paramiko
print('paramiko :', paramiko.__version__)

In [ ]:
import fabric
print('fabric :', fabric.__version__ if hasattr(fabric, '__version__') else 'installed')

---

## 3. Paramiko : premiere connexion

In [ ]:
# client = paramiko.SSHClient()
# client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
# client.connect('example.com', username='alice', key_filename='/home/alice/.ssh/id_ed25519')
print('Client configure (pas de vraie co dans notebook)')

---

## 4. Executer une commande

In [ ]:
# stdin, stdout, stderr = client.exec_command('uname -a')
# print(stdout.read().decode())
# client.close()
print('exec_command renvoie 3 file-like objects')

In [ ]:
# Gerer le code de retour
# stdout.channel.recv_exit_status()  # 0 si succes
print('recv_exit_status recupere le code')

---

## 5. SFTP : transfert de fichiers

In [ ]:
# sftp = client.open_sftp()
# sftp.put('local.txt', '/tmp/remote.txt')
# sftp.get('/tmp/remote.txt', 'copy.txt')
# sftp.listdir('/tmp')
# sftp.close()
print('SFTP : put, get, listdir, mkdir, remove, stat')

---

## 6. Gestion de cles

In [ ]:
# key = paramiko.Ed25519Key.from_private_key_file('/path/to/key')
# client.connect('host', username='u', pkey=key)
print('Ed25519 recommande ; RSA 4096 acceptable')

In [ ]:
# Utiliser ssh-agent
# agent = paramiko.Agent()
# keys = agent.get_keys()
print('ssh-agent evite de retaper la passphrase')

---

## 7. Fabric : connexions et groupes

In [ ]:
from fabric import Connection

# c = Connection('host.example.com', user='alice')
# result = c.run('uptime')
# print(result.stdout)
print('Connection + .run')

In [ ]:
from fabric import SerialGroup

# group = SerialGroup('web1', 'web2', 'web3')
# for conn, res in group.run('uname -a').items():
#     print(conn.host, '->', res.stdout.strip())
print('SerialGroup pour plusieurs hotes en sequence')

---

## 8. Sudo et context managers

In [ ]:
# c.sudo('apt update')   # demande password par config
# with c.cd('/var/www'): c.run('git pull')
# with c.prefix('source venv/bin/activate'): c.run('pytest')
print('cd, prefix, sudo sont des context managers')

---

## 9. Parallelisme

In [ ]:
from fabric import ThreadingGroup

# group = ThreadingGroup('web1', 'web2', 'web3')
# group.run('sudo systemctl restart nginx')
print('ThreadingGroup = parallele (thread par hote)')

---

## 10. Gestion d'erreurs

In [ ]:
from invoke import Responder
from fabric.exceptions import GroupException

# try:
#     group.run('cmd-peut-echouer', warn=True)
# except GroupException as e:
#     for host, err in e.result.failed.items():
#         print(host, err.exited)
print('warn=True : ne leve pas, renvoie result avec exited')

---

## 11. Logging et secrets

- **Jamais** hardcoder mots de passe et cles : utiliser `paramiko.Agent`, variables d'environnement, ou un coffre.
- Pour `sudo` : `config={'sudo': {'password': os.environ['SUDO_PASS']}}`
- Logger sans les credentials.
- Utiliser `ssh_config` pour centraliser les hotes.

---

## Synthese

| Operation | paramiko / fabric |
|---|---|
| Connect | `SSHClient().connect(host, username, key_filename=...)` |
| Exec | `client.exec_command('cmd')` |
| SFTP | `client.open_sftp().put/get/listdir` |
| Fabric run | `Connection(host).run('cmd')` |
| Multi-hote | `SerialGroup(*hosts).run(...)` |
| Parallele | `ThreadingGroup(*hosts).run(...)` |
| Sudo | `conn.sudo('cmd')` |
| cd | `with conn.cd('/path'):` |

### Regles a retenir

- **Paramiko** : bas niveau, flexible.
- **Fabric** : haut niveau, declaratif, basee sur paramiko.
- **Cles Ed25519** preferees.
- **Utiliser `~/.ssh/config`** pour factoriser hosts/users/keys.
- **`warn=True`** en batch pour ne pas stopper au premier echec.

---

## Exercices

### Exercice 1 (facile) -- Connect + uname

Ecrivez une fonction `remote_uname(host, user, key)` qui retourne le stdout de `uname -a`.

In [ ]:
import paramiko

# TODO

<details>
<summary>Voir la correction</summary>

```python
import paramiko

def remote_uname(host: str, user: str, key: str) -> str:
    c = paramiko.SSHClient()
    c.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    c.connect(host, username=user, key_filename=key)
    _, stdout, _ = c.exec_command('uname -a')
    out = stdout.read().decode()
    c.close()
    return out
```

</details>

### Exercice 2 (facile) -- SFTP upload

Fonction `upload(host, user, key, local, remote)` qui envoie un fichier.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
import paramiko

def upload(host: str, user: str, key: str, local: str, remote: str) -> None:
    c = paramiko.SSHClient()
    c.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    c.connect(host, username=user, key_filename=key)
    sftp = c.open_sftp()
    sftp.put(local, remote)
    sftp.close()
    c.close()
```

</details>

### Exercice 3 (moyen) -- Fabric group

Executez `df -h` sur 3 hotes `host1, host2, host3` en parallele et affichez les resultats.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from fabric import ThreadingGroup

group = ThreadingGroup('host1', 'host2', 'host3', user='alice')
results = group.run('df -h', hide=True)
for conn, res in results.items():
    print('---', conn.host)
    print(res.stdout)
```

</details>

### Exercice 4 (moyen) -- Deploiement minimal

Fonction `deploy(host, user, key)` qui :
1. `cd /var/www/app`
2. `git pull`
3. `pip install -r requirements.txt`
4. `sudo systemctl restart myapp`

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from fabric import Connection

def deploy(host: str, user: str, key: str) -> None:
    c = Connection(host, user=user, connect_kwargs={'key_filename': key})
    with c.cd('/var/www/app'):
        c.run('git pull')
        c.run('pip install -r requirements.txt')
    c.sudo('systemctl restart myapp')
```

</details>

### Exercice 5 (difficile) -- Verification multi-hote

Ecrivez une fonction qui execute `uptime` sur une liste de hotes en parallele, collecte les resultats et renvoie un dict `{host: charge_moyenne}`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from fabric import ThreadingGroup
import re

def check_uptime(hosts: list[str], user: str) -> dict[str, float]:
    group = ThreadingGroup(*hosts, user=user)
    results = group.run('uptime', hide=True, warn=True)
    loads: dict[str, float] = {}
    for conn, res in results.items():
        m = re.search(r'load average[s]?: ([\d.]+),', res.stdout)
        loads[conn.host] = float(m.group(1)) if m else -1.0
    return loads
```

</details>

---

## Ressources externes

- [Paramiko](https://www.paramiko.org/)
- [Fabric](https://www.fabfile.org/)
- [OpenSSH config](https://man.openbsd.org/ssh_config)